# Reproducing Gaddy & Klein silent speech recognition (open vocabulary)

Runs in **Google Colab with a GPU** (Runtime → Change runtime type → GPU). Tested on Colab with **Python 3.13** and a recent PyTorch on 2026-11-14; the compatibility fixes below are needed because the authors' code targets Python 3.9 / PyTorch 2.0.

1. Authors' code + submodules (phoneme alignments)
2. Dependencies + compatibility fixes (decoder, jiwer, PyTorch)
3. EMG dataset (Zenodo 4064409, 3.9 GB, CC BY 4.0) and pre-trained weights (Zenodo 7183877)
4. Evaluate on the 100-utterance silent test set: no training
5. Count the vocabulary
6. Simulate our hardware: band-limit EMG to 500 Hz and 250 Hz sampling, re-evaluate
7. Fine-tune the pre-trained model on 500 Hz band-limited data, re-evaluate

**Results so far (2026-11-14, no language model):** 1000 Hz 40.0 % WER · 500 Hz 88.0 % · 250 Hz 96.9 % (pre-trained model, not re-trained). Authors: ≈ 28 % with a language model.

Use `python -m pip` everywhere: in Colab `!pip` and `!python` can be different interpreters.

In [ ]:
!nvidia-smi -L || echo 'No GPU: switch the Colab runtime to GPU'
%cd /content
!git clone https://github.com/dgaddy/silent_speech.git
%cd /content/silent_speech
!git submodule init && git submodule update
!tar -xzf text_alignments/text_alignments.tar.gz
!git rev-parse HEAD
!python --version

## 1. Dependencies and compatibility fixes
- `ctcdecode` (authors' decoder) does not build on current Colab → replaced by a drop-in shim on `pyctcdecode` (same CTC beam search + KenLM idea; WER can differ slightly — a deviation).
- `kenlm` (language model bindings) does not build on Python 3.13 → decoding runs **without** the language model unless kenlm installs (worse WER than the paper).
- `jiwer==2.2.1` pulls a `Levenshtein` build that breaks on Python ≥ 3.12 → current `jiwer` (same `jiwer.wer`).
- `transformer.py`: newer PyTorch reads `self_attn.batch_first` → set to `False` (input is time-first; no change in computation).

In [ ]:
!python -m pip install -q absl-py librosa soundfile unidecode praat-textgrids noisereduce
!python -m pip install -q -U jiwer
!python -m pip install -q --no-deps pyctcdecode pygtrie
!python -m pip install -q https://github.com/kpu/kenlm/archive/master.zip || echo "kenlm not available: decoding without language model"
# KenLM language model used by the authors (DeepSpeech 0.6.1 release, ~945 MB)
!curl -L -o lm.binary https://github.com/mozilla/DeepSpeech/releases/download/v0.6.1/lm.binary
!ls -la lm.binary
!python -c "import sys, jiwer, pyctcdecode; print(sys.version.split()[0], 'wer test', jiwer.wer(['a b c'], ['a x c']))"
!python -c "import kenlm; print('kenlm OK')" || echo "kenlm missing"


In [ ]:
shim = r"""
# Drop-in replacement for parlance/ctcdecode, built on pyctcdecode.
import os, warnings
import torch
from pyctcdecode import build_ctcdecoder

class CTCBeamDecoder:
    def __init__(self, labels, model_path=None, alpha=0.0, beta=0.0, beam_width=100,
                 blank_id=0, log_probs_input=True, **kw):
        self.labels, self.blank_id, self.beam_width = list(labels), blank_id, beam_width
        vocab = ["" if i == blank_id else c for i, c in enumerate(self.labels)]
        ok = bool(model_path) and os.path.exists(model_path) and os.path.getsize(model_path) > 1e6
        lm = model_path if ok else None
        try:
            self.dec = build_ctcdecoder(vocab, kenlm_model_path=lm, alpha=alpha, beta=beta)
            print("ctcdecode shim: language model", "LOADED" if lm else "NOT FOUND (no LM)")
        except Exception as e:
            warnings.warn(f"LM not loaded ({e}); decoding WITHOUT language model")
            self.dec = build_ctcdecoder(vocab)
        self.index = {c: i for i, c in enumerate(self.labels) if i != blank_id}

    def decode(self, probs, seq_lens=None):
        x = probs.detach().float().cpu().numpy()
        B, T = x.shape[0], x.shape[1]
        out = torch.zeros(B, 1, T, dtype=torch.int)
        lens = torch.zeros(B, 1, dtype=torch.int)
        for b in range(B):
            text = self.dec.decode(x[b], beam_width=self.beam_width)
            ids = [self.index[c] for c in text if c in self.index][:T]
            out[b, 0, :len(ids)] = torch.tensor(ids, dtype=torch.int)
            lens[b, 0] = len(ids)
        return out, torch.zeros(B, 1), torch.zeros(B, 1, T, dtype=torch.int), lens
"""
open("ctcdecode.py", "w").write(shim)

src = open("transformer.py").read()
anchor = "    self.n_head = n_head\n"
if "self.batch_first" not in src:
    assert anchor in src, "transformer.py changed upstream: patch by hand"
    src = src.replace(anchor, anchor + "    self.batch_first = False  # newer PyTorch reads this; input is time-first\n", 1)
    open("transformer.py", "w").write(src)

src = open("read_emg.py").read()
anchor = "    x = np.concatenate([raw_emg_before, raw_emg, raw_emg_after], 0)\n"
patch = anchor + """    if os.environ.get('EMG_SIM_FS'):
        from scipy.signal import resample_poly
        sim = int(os.environ['EMG_SIM_FS'])
        n = x.shape[0]
        x = resample_poly(resample_poly(x, sim, 1000, axis=0), 1000, sim, axis=0)[:n]
"""
if "EMG_SIM_FS" not in src:
    assert anchor in src, "read_emg.py changed upstream: patch by hand"
    open("read_emg.py", "w").write(src.replace(anchor, patch))
!python -c "from ctcdecode import CTCBeamDecoder; print('shim ok')"
print("patched: ctcdecode shim, transformer.batch_first, read_emg EMG_SIM_FS")

## 2. EMG dataset (3.9 GB) and pre-trained weights

In [ ]:
!curl -L -o emg_data.tar.gz "https://zenodo.org/records/4064409/files/emg_data.tar.gz?download=1"
!echo "7f97d2182b896652999b1b2d0c69fd7b  emg_data.tar.gz" | md5sum -c -
!tar -xzf emg_data.tar.gz && rm emg_data.tar.gz
!ls emg_data

In [ ]:
import json, subprocess, urllib.request
rec = json.load(urllib.request.urlopen('https://zenodo.org/api/records/7183877'))
for f in rec['files']:
    name, url = f['key'], f['links']['self']
    print(name, round(f['size']/1e6, 1), 'MB')
    subprocess.run(['curl', '-L', '-o', name, url], check=True)
    if name.endswith(('.zip', '.tar.gz', '.tgz')):
        subprocess.run(['unzip', '-o', name] if name.endswith('.zip') else ['tar', '-xzf', name], check=True)
MODEL = subprocess.run("find . -name '*.pt' -not -path './hifi_gan/*' | head -1", shell=True,
                       capture_output=True, text=True).stdout.strip()
print('model file:', MODEL)

## 3. Evaluate on the silent test set (no training)
The WER is printed on screen (evaluation does not write log.txt). 2026-11-14: **0.400** without LM.

In [ ]:
!python recognition_model.py --evaluate_saved "$MODEL" --output_directory eval_full 2>&1 | grep -iE "wer|language model" 

## 4. How many distinct words are in the dataset?

In [ ]:
import glob, re, collections, json
counts = {}
for d in sorted(glob.glob('emg_data/*')):
    words = collections.Counter()
    for f in glob.glob(f'{d}/**/*_info.json', recursive=True):
        words.update(re.findall(r"[a-z']+", json.load(open(f)).get('text', '').lower()))
    counts[d] = words
    print(f'{d:40s} tokens={sum(words.values()):>8d}  distinct words={len(words):>6d}')
allw = sum(counts.values(), collections.Counter())
print('ALL tokens', sum(allw.values()), 'distinct words', len(allw))
print('20 most common:', allw.most_common(20))

## 5. Simulate our hardware (500 Hz and 250 Hz), same pre-trained model
Everything above the simulated Nyquist frequency is removed (1000 → fs → 1000 Hz). 2026-11-14: **500 Hz 0.880, 250 Hz 0.969** without LM.
This is the cost of feeding the model a band it never saw in training — not yet the cost of the hardware. Section 6 measures that.

In [ ]:
!EMG_SIM_FS=500 python recognition_model.py --evaluate_saved "$MODEL" --output_directory eval_500 2>&1 | grep -i "wer"
!EMG_SIM_FS=250 python recognition_model.py --evaluate_saved "$MODEL" --output_directory eval_250 2>&1 | grep -i "wer"


## 6. Fine-tune the pre-trained model on 500 Hz band-limited data
Starts from the authors' weights (`--start_training_from`), trains on the 500 Hz band-limited EMG, saves the best model after each epoch, stops after 60 minutes. Each finished epoch prints its validation WER. Then the saved model is evaluated on the test set at 500 Hz.

In [ ]:
!EMG_SIM_FS=500 timeout 60m python recognition_model.py --start_training_from "$MODEL" --output_directory ft_500 2>&1 | grep -E "finished epoch|train / dev"
!ls -la ft_500/
!EMG_SIM_FS=500 python recognition_model.py --evaluate_saved ft_500/model.pt --output_directory eval_ft_500 2>&1 | grep -i "wer"


## Results
| Step | Setting | WER (no LM) |
|---|---|---|
| 3 | pre-trained, 1000 Hz | 0.400 |
| 5 | pre-trained, band-limited 500 Hz | 0.880 |
| 5 | pre-trained, band-limited 250 Hz | 0.969 |
| 6 | fine-tuned at 500 Hz (60 min), 500 Hz | |
| 6 | minutes per epoch | |

Paste new numbers into `docs/decisions.md`.